# A Simple Calculator

This file shows the implementation of a *symbolic calculator* using `lark`. The grammar for the language implemented by this parser is as follows:
$$
\begin{array}{lcl}
  \texttt{stmnt}   & \rightarrow & \;\texttt{IDENTIFIER}\; \texttt{':='} \; \;\texttt{expr}\; \texttt{';'} \\
                   & \mid        & \;\texttt{expr}\; \texttt{';'} \\[0.2cm]   
  \texttt{expr}    & \rightarrow & \;\texttt{expr}\; \texttt{'+'} \; \texttt{product}  \\
                   & \mid        & \;\texttt{expr}\; \texttt{'-'} \; \texttt{product}  \\
                   & \mid        & \;\texttt{product}                                  \\[0.2cm]
  \texttt{product} & \rightarrow & \;\texttt{product}\; \texttt{'*'} \;\texttt{factor} \\
                   & \mid        & \;\texttt{product}\; \texttt{'/'} \;\texttt{factor} \\
                   & \mid        & \;\texttt{factor}                                   \\[0.2cm]
  \texttt{factor}  & \rightarrow &   \texttt{'('} \; \texttt{expr} \;\texttt{')'}      \\
                   & \mid        & \;\texttt{NUMBER}                                   \\
                   & \mid        & \;\texttt{IDENTIFIER}                               
\end{array}
$$

## Setup and Imports
We import `Lark` for parsing and `Token` to identify leaf nodes when walking the syntax tree.

In [ ]:
from lark import Lark, Token

## Specification of the Grammar

Unlike separated lexer and parser libraries, `lark` allows us to define both lexical tokens and structural rules natively in a single *extended Backus-Naur Form* (EBNF) string. Let us break down the components of our grammar definition:

* `?start: stmnt`: This tells Lark that the entry point for our grammar is a statement (`stmnt`). 
   The `?` prefix instructs the parser to inline this rule if it only has one child, preventing unnecessary nesting in our syntax tree.
* `stmnt`, `expr`, `product`, `factor`: These variables define the hierarchical structure of our mathematical expressions, 
   natively enforcing operator precedence (e.g., multiplication in `product` happens before addition in `expr`). 
   The `?` prefix is used on all of these to keep our resulting tree flat when intermediate rules simply pass a single value up the chain.
* **Aliases** (`-> assign`, `-> add`, etc.): When a rule can branch into multiple different operations (like `expr` being either addition or 
   subtraction), we use the `->` operator to assign a specific name to that branch. This guarantees that our resulting tree nodes have clear, 
   identifiable names representing the operation, which is crucial when we build our Abstract Syntax Tree (AST).
* **Lexical Imports** (`%import common...`): Instead of manually writing regular expressions for common tokens, we import standard terminal
   definitions provided by Lark. `
   * CNAME` parses standard variable names (letters, numbers, underscores) and is aliased to `IDENTIFIER`.
   * `NUMBER` handles both integers and floating-point numbers. 
   * `WS` handles whitespace.
* `%ignore WS`: This directs the lexer to completely discard spaces and tabs so they do not clutter our parsing logic.

In [ ]:
calc_grammar = """
?start: stmnt

?stmnt: IDENTIFIER ":=" expr ";" -> assign
      | expr ";"                 -> expr_stmnt

?expr: expr "+" product          -> add
     | expr "-" product          -> sub
     | product

?product: product "*" factor     -> mul
        | product "/" factor     -> div
        | factor

?factor: "(" expr ")"
       | NUMBER                  -> number
       | IDENTIFIER              -> var

// Lexical definitions
%import common.CNAME -> IDENTIFIER
%import common.NUMBER
%import common.WS

ASSIGN: ":="
SEMI:   ";"

// Ignore whitespace
%ignore WS
"""

Instantiate the parser using the LALR algorithm. We do not use a Transformer here.

In [ ]:
parser = Lark(calc_grammar, parser='lalr')

## Generating the Abstract Syntax Tree (ADT)

Rather than relying on library-specific objects to evaluate our code, we will manually walk the `lark.Tree` generated by the parser and convert it into a pure Python Abstract Data Type (ADT) represented by nested tuples.

For example, the string `x := y + z;` will be parsed into the ADT `(':=', 'x', ('+', ('var', 'y'), ('var', 'z')))`.

In [ ]:
def ast_to_tuples(node):
    # Base case: If it's a leaf node (a Token), return its string value
    if isinstance(node, Token):
        return str(node)
    # Map Lark's node names to our preferred operator strings
    operator_map = {
        'assign':     ':=',
        'expr_stmnt': 'eval',
        'add':        '+',
        'sub':        '-',
        'mul':        '*',
        'div':        '/',
        'number':     'num',
        'var':        'var'
    }   
    # Get the operator string (default to the raw node data if not in the map)
    op = operator_map.get(node.data, node.data)
    # Recursively convert all children
    children = [ast_to_tuples(child) for child in node.children]
    # Return the nested tuple
    return (op, *children)

## Evaluating the Abstract Syntax Tree

Now that we have our pure ADT made of nested tuples, we can write our own recursive evaluation function. The variable `Names2Values` is a global dictionary that stores the values associated with the variables we define.

In [ ]:
Names2Values = {}

In [ ]:
def evaluate_adt(adt):
    match adt:
        case ('num', value):
            return float(value)
        case ('var', var_name):
            if var_name not in Names2Values:
                print(f"Syntax Error: Variable '{var_name}' is undefined.")
                return 0.0
            return Names2Values[var_name]
        case (':=', var_name, expr):
            Names2Values[var_name] = evaluate_adt(expr)         
        case ('eval', expr):
            print(evaluate_adt(expr))
        case (('+' | '-' | '*' | '/') as op, left, right):
            val_left  = evaluate_adt(left )
            val_right = evaluate_adt(right)
            match op:
                case '+': return val_left + val_right
                case '-': return val_left - val_right
                case '*': return val_left * val_right
                case '/': return val_left / val_right
        case _:
            raise ValueError(f"Unknown ADT structure: {adt}")

## Execution

The REPL (Read-Eval-Print Loop) takes the input string, parses it into Lark's tree, converts it into our nested tuples, and then evaluates it.

In [ ]:
def main():
    while True:
        try:
            s = input('calc> ')
            if s == '':
                break
            # 1. Parse the string into the Lark Tree
            raw_tree = parser.parse(s)
            # 2. Convert it to a nested tuple
            adt_tuples = ast_to_tuples(raw_tree)           
            # 3. Evaluate the nested tuple 
            evaluate_adt(adt_tuples)
        except Exception as e:
            print(f"Error: {e}")

In [ ]:
main()

Check the populated memory map:

In [ ]:
Names2Values

We can even get the parser states.

In [ ]:
from lark import Lark
from lark.parsers.lalr_analysis import LALR_Analyzer

In [ ]:
# 1. Load the autoreload extension so local file changes are instantly recognized
%load_ext autoreload
%autoreload 2

In [ ]:
# 2. Import your custom library
from formal_methods_viz import print_lalr1_states

In [ ]:
# 3. Define the token map for the grammar above
token_map = {
    'ASSIGN': "':='",
    'SEMI': "';'",
    'SEMICOLON': "';'",
    'IDENTIFIER': "'id'",
    'NUMBER': "'num'",
    
    # Mathematical Operators
    'ADDOP': "'+' or '-'",
    'MULOP': "'*' or '/'",
    'PLUS': "'+'",
    'MINUS': "'-'",
    'STAR': "'*'",
    'SLASH': "'/'",
    
    # Parentheses
    'LPAR': "'('",
    'RPAR': "')'"
}

In [ ]:
# 4. Generate the states!
print_lalr1_states(parser, token_map)